## 6. Arama Motoru ile Sorgu (SerpAPI + Agent)
* Amaç: LLM’in internette araştırma yapma yeteneğini göstermek.
* Senaryo: Kullanıcı güncel bir soruya cevap arar, model SerpAPI aracını kullanır.
* Kazandırılan: Web tabanlı ajan etkileşimi.
* Kullanılan Bileşenler: SerpAPIWrapper, Agent, Tool

In [ ]:
# ============================================================
# 1) KURULUM
# ============================================================
!pip install -q "langchain>=0.2" "langchain-community>=0.2" \
               "langchain-experimental>=0.0.64" \
               duckduckgo-search \
               transformers accelerate sentencepiece

In [ ]:


# ============================================================
# 2) İÇE AKTARIMLAR
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline

from langchain.agents import initialize_agent, AgentType
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_experimental.tools.python.tool import PythonREPLTool  # İsteğe bağlı

# ============================================================
# 3) LLM MODELİ: Açık kaynak Qwen
# ============================================================
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # Alternatif: TinyLlama 1.1B

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

gen_pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.2,
    top_p=0.9,
    repetition_penalty=1.05,
)

llm = HuggingFacePipeline(pipeline=gen_pipe)

# ============================================================
# 4) ARAMA ARACI: DuckDuckGo
# ============================================================
search_tool = DuckDuckGoSearchRun(name="WebSearch")

# (İsteğe Bağlı) Python hesaplama aracı
python_tool = PythonREPLTool(name="Python")

# ============================================================
# 5) AGENT TANIMI
# ============================================================
agent = initialize_agent(
    tools=[search_tool],  # gerekirse: + python_tool
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
)

# ============================================================
# 6) SORU & YÜRÜTME
# ============================================================
question = "Türkiye Cumhuriyet Merkez Bankası başkanı kimdir?"

print(f"\n=== Soru ===\n{question}")
response = agent.run(question)

print("\n=== Cevap ===")
print(response)
